# Demo User SQL Generator

Cleaned public-repository version of the IE332 user seed generator. The original notebook was used to create `User` table insert, update, and reset scripts for assignment testing. This version uses generic demo accounts and does not store generated passwords in notebook outputs.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

In [ ]:
@dataclass(frozen=True)
class DemoUser:
    full_name: str
    username: str
    role: str


VALID_ROLES = {"SeniorManager", "SupplyChainManager"}

DEMO_USERS = [
    DemoUser("Demo Senior Manager", "demo_sem", "SeniorManager"),
    DemoUser("Demo Supply Chain Manager", "demo_scm", "SupplyChainManager"),
    DemoUser("Recruiter Review Senior", "review_sem", "SeniorManager"),
    DemoUser("Recruiter Review SCM", "review_scm", "SupplyChainManager"),
    DemoUser("Analytics Reviewer", "analytics_sem", "SeniorManager"),
    DemoUser("Operations Reviewer", "operations_scm", "SupplyChainManager"),
]


def escape_sql(value: str) -> str:
    return value.replace("\\", "\\\\").replace("'", "''")


def validate_users(users: list[DemoUser]) -> None:
    usernames = [user.username for user in users]
    duplicates = sorted({name for name in usernames if usernames.count(name) > 1})

    if duplicates:
        raise ValueError(f"Duplicate usernames: {duplicates}")

    invalid_roles = sorted({user.role for user in users if user.role not in VALID_ROLES})
    if invalid_roles:
        raise ValueError(f"Invalid roles: {invalid_roles}")


validate_users(DEMO_USERS)

In [ ]:
SQL_HEADER = """-- Demo-only user seed data for local development.
-- Before running this file, set a local SQL session variable:
-- SET @demo_user_password := 'your-local-demo-password';
"""


def build_insert_sql(users: list[DemoUser], password_variable: str = "@demo_user_password") -> str:
    rows = []
    for user in users:
        rows.append(
            "    SELECT "
            f"'{escape_sql(user.full_name)}' AS FullName, "
            f"'{escape_sql(user.username)}' AS Username, "
            f"MD5({password_variable}) AS Password, "
            f"'{escape_sql(user.role)}' AS Role"
        )

    joined_rows = "\n    UNION ALL\n".join(rows)
    return (
        SQL_HEADER
        + "\nINSERT INTO User (FullName, Username, Password, Role)\n"
        + "SELECT FullName, Username, Password, Role\n"
        + "FROM (\n"
        + joined_rows
        + "\n) AS demo_users\n"
        + f"WHERE {password_variable} IS NOT NULL;\n"
    )


def build_update_sql(users: list[DemoUser], password_variable: str = "@demo_user_password") -> str:
    statements = [SQL_HEADER]
    for user in users:
        statements.append(
            "\nUPDATE User\n"
            f"SET FullName = '{escape_sql(user.full_name)}',\n"
            f"    Password = MD5({password_variable}),\n"
            f"    Role = '{escape_sql(user.role)}'\n"
            f"WHERE Username = '{escape_sql(user.username)}'\n"
            f"  AND {password_variable} IS NOT NULL;\n"
        )
    return "".join(statements)


def build_reset_sql() -> str:
    return "-- Local development reset only.\n-- This removes every row from the User table.\n\nTRUNCATE TABLE User;\n"

In [ ]:
def write_sql_files(output_dir: Path = Path(".")) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "user_insert.sql").write_text(build_insert_sql(DEMO_USERS), encoding="utf-8")
    (output_dir / "user_update.sql").write_text(build_update_sql(DEMO_USERS), encoding="utf-8")
    (output_dir / "user_reset.sql").write_text(build_reset_sql(), encoding="utf-8")

In [ ]:
# Uncomment to regenerate the SQL files in the current notebook directory.
# write_sql_files()